# Modelo Yeast9

In [1]:
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
import numpy as np

In [2]:
model_yeast9 = read_sbml_model("yeast-GEM.xml")

In [ ]:
# Imprimir información básica del modelo
print("Reacciones:", len(model_yeast9.reactions))
print("Reacciones de exchange:", len(model_yeast9.exchanges))
print("Metabolitos:", len(model_yeast9.metabolites))
print("Genes:", len(model_yeast9.genes))

In [ ]:
contador_exportaciones = 0
contador_importaciones = 0
for rxn in model_yeast9.exchanges:
    if rxn.lower_bound < 0:  # Solo reacciones de importación
        contador_importaciones += 1
        print(f"{rxn.id}: {rxn.name} (límite: {rxn.lower_bound})")
    if rxn.upper_bound > 0:  # Solo reacciones de exportación
        contador_exportaciones += 1
        
print(f"Total de reacciones de importación: {contador_importaciones}")
print(f"Total de reacciones de exportación: {contador_exportaciones}")

In [2]:
def imprimir_precios_sombra(solucion, model):
    for met_id, value in solucion.shadow_prices.items():
        if abs(value) > 1:
            met = model.metabolites.get_by_id(met_id)
            print(f"{met_id} | {met.name} | shadow price = {value:.3f}")
            
            
def imprimir_costos_reducidos(solucion, model):
    print("Reacciones con costos reducidos positivos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value > 1e-6 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")
            
    print("\nReacciones con costos reducidos negativos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value < -1 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")

Los precios sombra son por metabolito. 

Si el precio sombra de un metabolito es alto (en valor absoluto), significa que: 
- Ese metabolito es limitante para el crecimiento. 
- Una pequeña disponibilidad extra mejoraría la biomasa.

Los costos reducidos son cantidades asociadas a cada reacción (flujo) que indican cuánto tendría que mejorar esa reacción en términos del objetivo (por ejemplo, biomasa) para que pase de estar inactiva (flujo cero) a activarse en la solución óptima. 

En términos prácticos, si estás maximizando crecimiento y una reacción tiene reduced cost positivo:
-  activarla aumentaría la biomasa, pero actualmente está impedida por alguna restricción (como un bound cerrado).

si es negativo:
- activarla disminuiría la biomasa y por eso el modelo no la usa.

y si es cero:
- la reacción ya es óptima o indiferente en la solución actual. 

En resumen, los reduced costs te dicen qué reacciones “querrían” activarse para mejorar el objetivo y cuáles están correctamente apagadas bajo las condiciones actuales.

## FBA modelo tal cual:

In [ ]:
with model_yeast9:
    sol1 = model_yeast9.optimize()
    print(f"Solución optimización: {sol1.objective_value}")
    fva_res = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)
    imprimir_precios_sombra(sol1, model_yeast9)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol1, model_yeast9)
    

In [ ]:
for rxn_id, fila in fva_res.iterrows():
    if fila["minimum"] < 0:
        rxn = model_yeast9.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

## FBA modelo con glucosa en -10

In [ ]:
with model_yeast9:
    model_yeast9.exchanges.get_by_id("r_1714").lower_bound = -10.0
    sol2 = model_yeast9.optimize()
    print(f"Solución optimización: {sol2.objective_value}")
    fva_res2 = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)
    imprimir_precios_sombra(sol2, model_yeast9)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol2, model_yeast9)

In [ ]:
for rxn_id, fila in fva_res2.iterrows():
    if fila["minimum"] < 0:
        rxn = model_yeast9.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

## FBA modelo anaeróbico changes 2 y 4:

In [3]:
model = read_sbml_model("yeast-GEM.xml")

In [4]:
from modificacion_yeast9 import modify_yeast9_anaerobic

In [5]:
model_anaerobic = modify_yeast9_anaerobic(model)

Removed from r_4598: ['s_3714', 's_1198', 's_1203', 's_1207', 's_1212', 's_0529']
Not found: []


In [6]:
model_anaerobic.exchanges.get_by_id("r_1992").lower_bound = 0.0

In [7]:
with model_anaerobic:
    sol3 = model_anaerobic.optimize()
    print(f"Solución optimización: {sol3.objective_value}")
    fva_res3 = flux_variability_analysis(model_anaerobic, model_anaerobic.exchanges, fraction_of_optimum=0.9, processes = 4)
    # imprimir_precios_sombra(sol3, model_anaerobic)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol3, model_anaerobic)

Solución optimización: 0.0

 -----------------------
Reacciones con costos reducidos positivos

Reacciones con costos reducidos negativos
r_1992 | oxygen exchange | shadow price = -21.442
r_1994 | palmitoleate exchange | shadow price = -21.442
r_2028 | pyridoxine exchange | shadow price = -10.721


In [8]:
for rxn_id, fila in fva_res3.iterrows():
    if fila["minimum"] < 0:
        rxn = model_anaerobic.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

r_1654 - ammonium exchange: mínimo de -1.30000
r_1714 - D-glucose exchange: mínimo de -1.00000
r_1832 - H+ exchange: mínimo de -1.30000
r_2005 - phosphate exchange: mínimo de -2.60000
r_2060 - sulphate exchange: mínimo de -0.45882
r_2100 - water exchange: mínimo de -0.35455


## FBA modelo anaeróbico changes 2 3 y 4:

In [9]:
with model_anaerobic:
    for factores_anaerobicos in ["r_1757","r_1915","r_1994","r_2106","r_2134","r_2137","r_2189"]:
        if factores_anaerobicos in model_anaerobic.reactions:
            model_anaerobic.reactions.get_by_id(factores_anaerobicos).lower_bound = -1000
    
    sol4 = model_anaerobic.optimize()
    print(f"Solución optimización: {sol4.objective_value}")
    fva_res4 = flux_variability_analysis(model_anaerobic, model_anaerobic.exchanges, fraction_of_optimum=0.9, processes=4)
    # imprimir_precios_sombra(sol4, model_anaerobic)
    print("\n -----------------------")
    imprimir_costos_reducidos(sol4, model_anaerobic)
            

Solución optimización: 0.01296449057516892

 -----------------------
Reacciones con costos reducidos positivos

Reacciones con costos reducidos negativos
r_1992 | oxygen exchange | shadow price = -2.314
r_2028 | pyridoxine exchange | shadow price = -1.147


In [10]:
for rxn_id, fila in fva_res4.iterrows():
    if fila["minimum"] < 0:
        rxn = model_anaerobic.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")

r_1654 - ammonium exchange: mínimo de -0.20415
r_1714 - D-glucose exchange: mínimo de -1.00000
r_1757 - ergosterol exchange: mínimo de -0.00048
r_1832 - H+ exchange: mínimo de -0.19467
r_1861 - iron(2+) exchange: mínimo de -0.00000
r_1967 - nicotinate exchange: mínimo de -0.00000
r_1994 - palmitoleate exchange: mínimo de -0.00121
r_2005 - phosphate exchange: mínimo de -0.43736
r_2020 - potassium exchange: mínimo de -0.00005
r_2049 - sodium exchange: mínimo de -0.00005
r_2060 - sulphate exchange: mínimo de -0.04688
r_2100 - water exchange: mínimo de -0.15285
r_2106 - zymosterol exchange: mínimo de -0.02516
r_2137 - ergosta-5,7,22,24(28)-tetraen-3beta-ol exchange: mínimo de -0.78000
r_2189 - oleate exchange: mínimo de -0.00040
r_4593 - chloride exchange: mínimo de -0.00002
r_4594 - Cu2(+) exchange: mínimo de -0.00001
r_4595 - Mn(2+) exchange: mínimo de -0.00004
r_4596 - Zn(2+) exchange: mínimo de -0.00001
r_4597 - Mg(2+) exchange: mínimo de -0.00002
r_4600 - Ca(2+) exchange: mínimo de -0

## FBA modelo anaeróbico con medio enológico

In [10]:
medio_enologico = [
    # "r_1714","r_1709",      # CARBONO: D-glucose, D-fructose
    "r_1687","r_1552",      # ÁCIDOS ORGÁNICOS: citrate, (S)-malate
    "r_2005","r_2060",      # SALES/IONES: phosphate, sulphate
    "r_2020","r_2049",      # potassium, sodium 
    "r_4593","r_4600",      # chloride, Ca2+
    "r_4597","r_4595",      # Mg2+, Mn2+
    "r_4596","r_4594",      # Zn2+, Cu2+
    "r_1947","r_1967",      # VITAMINAS: myo-inositol, nicotinate
    "r_1548","r_2067",      # (R)-pantothenate, thiamine(1+)
    # "r_2028",               # pyridoxine, (SI LA DEJO NO SE PRODUCE ETANOL, cambios 2 y 4)
    "r_1671",               # biotin
    "r_1757","r_2189",      # FACTORES ANAERÓBICOS: ergosterol, oleate
    "r_1654",               # ammonium
    # r_1757","r_1915","r_1994","r_2106","r_2134","r_2137","r_2189
    "r_1915","r_1994",      # OTROS FACTORES ANAERÓBICOS: lanosterol, palmitoleate
    "r_2106","r_2134",      # zymosterol, 14-demethyllanosterol
    "r_2137",               # ergosta-5,7,22,24(28)-tetraen-3beta-ol
    "r_1873","r_1879",      # AMINOÁCIDOS: alanine, arginine
    "r_1881","r_1889",      # aspartate, glutamate
    "r_1891","r_1893",      # glutamine, histidine
    "r_1897","r_1899",      # isoleucine, leucine
    "r_1900","r_1902",      # lysine, methionine
    "r_1903","r_1906",      # phenylalanine, proline
    "r_1911","r_1912",      # threonine, tryptophan
    "r_1913","r_1914",      # tyrosine, valine
    "r_1810",               # glycine
    # EXTRAS PARA QUE FUNCIONE MODELO:
    "r_1832",               # H+
    "r_1861",               # iron(2+)
    "r_2100"                # water
]

In [ ]:
with model_anaerobic:
    for componente in medio_enologico:
        if componente in model_anaerobic.exchanges:
            model_anaerobic.reactions.get_by_id(componente).lower_bound = -1000
    
    print(f'Lower bound de glucosa: {model_anaerobic.exchanges.get_by_id("r_1714").lower_bound}')

    sol5 = model_anaerobic.optimize()
    print(f"Solución optimización: {sol5.objective_value}")
    fva_res5 = flux_variability_analysis(model_anaerobic, model_anaerobic.exchanges, fraction_of_optimum=0.9, processes = 4)
    #imprimir_precios_sombra(sol5, model_anaerobic)
    print("\n -----------------------")
    # imprimir_costos_reducidos(sol5, model_anaerobic)

Lower bound de glucosa: -1.0
Solución optimización: 10.055168034945527

 -----------------------


In [9]:
for rxn_id, fila in fva_res5.iterrows():
    if fila["minimum"] < 0:
        rxn = model_anaerobic.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: mínimo de {fila['minimum']:.5f}")
    
print("\n -----------------------")

r_1552 - (S)-malate exchange: mínimo de -1000.00000
r_1654 - ammonium exchange: mínimo de -1.00000
r_1687 - citrate exchange: mínimo de -1000.00000
r_1714 - D-glucose exchange: mínimo de -1.00000
r_1757 - ergosterol exchange: mínimo de -0.37540
r_1810 - L-glycine exchange: mínimo de -955.03960
r_1832 - H+ exchange: mínimo de -1000.00000
r_1861 - iron(2+) exchange: mínimo de -0.00028
r_1873 - L-alanine exchange: mínimo de -1000.00000
r_1879 - L-arginine exchange: mínimo de -1000.00000
r_1881 - L-aspartate exchange: mínimo de -1000.00000
r_1889 - L-glutamate exchange: mínimo de -1000.00000
r_1891 - L-glutamine exchange: mínimo de -1000.00000
r_1893 - L-histidine exchange: mínimo de -0.76577
r_1897 - L-isoleucine exchange: mínimo de -802.46276
r_1899 - L-leucine exchange: mínimo de -1000.00000
r_1900 - L-lysine exchange: mínimo de -3.30565
r_1902 - L-methionine exchange: mínimo de -1000.00000
r_1903 - L-phenylalanine exchange: mínimo de -816.71019
r_1906 - L-serine exchange: mínimo de -10

Límites inferiores que no llegan a -1000:

r_1654 - ammonium exchange: mínimo de -590.98858

r_1714 - D-glucose exchange: mínimo de -1.00000

r_1757 - ergosterol exchange: mínimo de -0.37540

r_1810 - L-glycine exchange: mínimo de -955.03960

r_1861 - iron(2+) exchange: mínimo de -0.00028

r_1893 - L-histidine exchange: mínimo de -0.76577

r_1897 - L-isoleucine exchange: mínimo de -802.46276

r_1900 - L-lysine exchange: mínimo de -3.30565

r_1903 - L-phenylalanine exchange: mínimo de -816.71019

r_1912 - L-tryptophan exchange: mínimo de -815.61350

r_1915 - lanosterol exchange: mínimo de -0.00000

r_1947 - myo-inositol exchange: mínimo de -22.37638

r_1994 - palmitoleate exchange: mínimo de -0.93789

r_2005 - phosphate exchange: mínimo de -136.17363

r_2020 - potassium exchange: mínimo de -0.03650

r_2049 - sodium exchange: mínimo de -0.03992

r_2060 - sulphate exchange: mínimo de -67.07766

r_2067 - thiamine(1+) exchange: mínimo de -0.00001

r_2106 - zymosterol exchange: mínimo de -366.28930

r_2189 - oleate exchange: mínimo de -0.30994

r_4593 - chloride exchange: mínimo de -0.01297

r_4594 - Cu2(+) exchange: mínimo de -0.00663

r_4595 - Mn(2+) exchange: mínimo de -0.02745

r_4596 - Zn(2+) exchange: mínimo de -0.00752

r_4597 - Mg(2+) exchange: mínimo de -0.01249

r_4600 - Ca(2+) exchange: mínimo de -0.00218

In [12]:
for rxn_id, fila in fva_res5.iterrows():
    if fila["maximum"] > 0:
        rxn = model_anaerobic.reactions.get_by_id(rxn_id)
        print(f"{rxn.id} - {rxn.name}: máximo de {fila['maximum']:.5f}")

r_1546 - (R)-lactate exchange: máximo de 33.44833
r_1547 - (R)-mevalonate exchange: máximo de 183.14465
r_1549 - (R,R)-2,3-butanediol exchange: máximo de 580.74471
r_1550 - (S)-3-methyl-2-oxopentanoate exchange: máximo de 1000.00000
r_1553 - 1-(sn-glycero-3-phospho)-1D-myo-inositol exchange: máximo de 22.29889
r_1572 - 2-isopropylmalate exchange: máximo de 653.59828
r_1577 - 2-methylbutanal exchange: máximo de 750.24905
r_1580 - 2-methylbutanol exchange: máximo de 811.80414
r_1581 - 2-methylbutyl acetate exchange: máximo de 592.43189
r_1586 - 2-oxoglutarate exchange: máximo de 1000.00000
r_1589 - 2-phenylethanol exchange: máximo de 815.31828
r_1598 - 3-methylbutanal exchange: máximo de 750.24905
r_1604 - 4-aminobenzoate exchange: máximo de 12.54313
r_1630 - 9H-xanthine exchange: máximo de 16.72417
r_1631 - acetaldehyde exchange: máximo de 1000.00000
r_1634 - acetate exchange: máximo de 1000.00000
r_1641 - adenosine 3',5'-bismonophosphate exchange: máximo de 7.16750
r_1650 - trehalose e

Producción de etanol puede llegar a 1000